## Tavily搜索引擎
## ReAct Agent工作流程 model -> tool -> model -> tool -> model

In [6]:
from langchain.agents import create_agent
from langchain_classic.chains.summarize.refine_prompts import prompt_template
from langchain_tavily import TavilySearch
from langchain_core.tools import StructuredTool
from langchain_community.tools import TavilySearchResults
from langchain_core.tools import Tool
from langchain_openai import ChatOpenAI
import os
import dotenv

dotenv.load_dotenv()
llm=ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    temperature=0,
    model="gpt-4o-mini",
)

#获取Tavily搜索工具的实例（父类是BaseTool）
search_tool = TavilySearch(
    tavily_api_key=os.getenv("TAVILY_API_KEY"),
    max_results=3,
)


#LangGraph 风格的 agent
graph=create_agent(
    tools=[search_tool],
    model=llm,
    system_prompt="You are a helpful assistant",
)

inputs={
    "messages":[
        {"role": "user", "content": "查询青岛明天的天气情况"}
    ]
}

# response=graph.invoke(inputs)
# print(response)

#迭代式输出，观察agent 进度
for chunk in graph.stream(inputs, stream_mode="updates"):
    print(chunk)
"""
Action 1：第一次搜索
Observation 1：失败 + 错误建议
Action 2：修改参数后再次搜索
Observation 2：拿到天气结果
Final Answer：输出总结
"""


{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 1201, 'total_tokens': 1232, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eb37e061ec', 'id': 'chatcmpl-DVIQAXoPuYNkJ3VVTeiLTGSmFunVd', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d96ca-8962-7b02-b247-da0ff3e670f9-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': '青岛 明天 天气', 'time_range': 'day', 'topic': 'general'}, 'id': 'call_cwFe88xyFj2AxT2V6Ku9Fo8S', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1201, 'output_tokens': 31, 'total_tokens': 1232, 'input_token_details': {'audio': 0, 'cache_re

## 多工具的使用
## Tool 和 @tool 的关系
它们本质目标一样，都是：把函数变成 agent 可调用工具

Tool()适合你已经有一个现成函数，手动包装：, @tool 适合自己定义一个函数，然后自动包装成工具对象

In [5]:
from langchain.agents import create_agent
from langchain_tavily import TavilySearch
from langchain_openai import ChatOpenAI
import os
from langchain_core.tools import Tool
import dotenv
from langchain_experimental.utilities.python import PythonREPL


dotenv.load_dotenv()
llm=ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    temperature=0,
    model="gpt-4o-mini",
)

#获取Tavily搜索工具的实例
search_tool = TavilySearch(
    tavily_api_key=os.getenv("TAVILY_API_KEY"),
    max_results=3,
)

#定义计算工具
python_repl=PythonREPL()
#Tool 的核心作用就是把一个普通函数包装成“agent 可以调用的工具对象
cal_tool=Tool(
    name="Calculator",
    func=python_repl.run,
    description="用于执行数学计算，例如算百分比变化"
)

#LangGraph 风格的 agent
graph=create_agent(
    tools=[search_tool, cal_tool],
    model=llm,
    system_prompt="""You are a helpful assistant.
For any numerical calculation, percentage change, or arithmetic, always use the Calculator tool instead of mental math."""
)

inputs={
    "messages":[
        {"role": "user", "content": "当前比亚迪股价是多少？比去年上涨了百分之几？"}
    ]
}

#迭代式输出，观察agent 进度
for chunk in graph.stream(inputs, stream_mode="updates"):
    print(chunk)



{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 1261, 'total_tokens': 1286, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eb37e061ec', 'id': 'chatcmpl-DVWhYX5GXH0nzw10CS1HJyz0urQuq', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d9a10-2ce8-7b42-8361-d9f06c93146c-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': '比亚迪股价', 'topic': 'finance'}, 'id': 'call_BE79mQKJhGA17dKQzRRiE2fK', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1261, 'output_tokens': 25, 'total_tokens': 1286, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_d

Python REPL can execute arbitrary code. Use with caution.


{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 3981, 'total_tokens': 4011, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1152}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eb37e061ec', 'id': 'chatcmpl-DVWheYCBJou6CXIw5OykcqMWf5MCD', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d9a10-4282-7a22-98a1-7b0053a49e79-0', tool_calls=[{'name': 'Calculator', 'args': {'__arg1': '(97.3 - 80) / 80 * 100'}, 'id': 'call_zhWPAeeoBthbNcG0Gf3u8ulA', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 3981, 'output_tokens': 30, 'total_tokens': 4011, 'input_token_details': {'audio': 0, 'cache_read': 1152}, 'output_token_

## 自定义工具

In [6]:
def simple_calculator(query: str) -> str:
    print("调用工具成功")

    return str(eval(query))

cal_tool=Tool(
    name="Math_Calculator",
    func=simple_calculator,
    description="用于执行数学计算,输入是纯数学表达式，；例如“3+1"
)
graph=create_agent(
    tools=[cal_tool],
    model=llm,
    system_prompt="You are a helpful assistant"
)
inputs={
    "messages":[
        {"role": "user", "content": "3+1"}
    ]
}
for chunk in graph.stream(inputs, stream_mode="updates"):
    print(chunk)

{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 75, 'total_tokens': 96, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eb37e061ec', 'id': 'chatcmpl-DVX4mSLiCgg3NeAESFtpGPlythanl', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d9a26-22e9-7b23-8586-8cf1ea205ce3-0', tool_calls=[{'name': 'Math_Calculator', 'args': {'__arg1': '3+1'}, 'id': 'call_ZZY22rtHu3A8hOeJLoUexenX', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 75, 'output_tokens': 21, 'total_tokens': 96, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reas

## Function Calling Agent
create_tool_calling_agent(
llm: BaseLanguageModel,
tools: Sequence[BaseTool],
prompt: ChatPromptTemplate,
*,
message_formatter: MessageFormatter = format_to_tool_messages
) -> Runnable

In [8]:
from langchain_classic.agents import (
    AgentExecutor,
    create_tool_calling_agent,
    tool,
)
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder


prompt_template=ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant"),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),#########必须的参数agent_scratchpad#######
    ]
)

@tool
def magic_function(input: int) -> int:
    """Applies a magic function to an input."""
    return input + 2

agent=create_tool_calling_agent(
    llm=llm,
    tools=[magic_function],
    prompt=prompt_template,
)

## 获取AgentExecutor的实例
agent_executor=AgentExecutor(agent=agent, tools=[magic_function])

response=agent_executor.invoke({
    "input": "我叫什么名字？what is the value of magic_function(3)?",
    "chat_history": ["我叫小林"]
})
print(response)
#AgentExecutor.invoke() 返回的不是底层聊天模型原始消息对象，而是一个链式/执行器风格的结果字典

{'input': '我叫什么名字？what is the value of magic_function(3)?', 'chat_history': ['我叫小林'], 'output': '你叫小林。调用 `magic_function(3)` 的结果是 5。'}
